# 10. Random fields

A random field is a function whose values are drawn at random with a
prescribed statistical structure. The fields here are Gaussian with
zero mean, a chosen standard deviation, and a correlation between
nearby points that falls off over a length scale. Fields of Matern
type are the standard family for this: besides the length scale they
have a smoothness parameter `nu`, and `planetmodel.randomfield` builds
them on a ball, on a spherical shell, or on the layers of a skeleton.
The construction is the one through a stochastic partial differential
equation: the covariance is a power of an elliptic operator, which is
discretised on a radial spectral-element mesh, one spherical-harmonic
degree at a time.

Three samplers are provided. `RadialGRF` draws a field of the radius
alone. `LayeredGRF` draws an independent field on each chosen layer of
a model and returns fields the model can hold. `SphericalGRF` draws a
field on a shell, delivered as spherical-harmonic coefficients that
are functions of radius, and its `to_field` turns one into a field a
model can hold. Underneath them are the operator family and its
eigenfunctions, which are public too, so that other code (pygeoinf,
for one) can build function spaces and probability measures on them.

This tutorial plots, so it needs the `plot` extra (matplotlib). The
figure is written to `examples/figures/`.

In [ ]:
# Generated from examples/tutorials/10_random_fields.py.
# Edit the script, not this notebook.  On Google Colab the library is
# installed first, from GitHub, with the extras this tutorial needs.
# Everywhere, the notebook is given the script's place, so that the files
# it writes and the data it reads are found as the script finds them; its
# figures are shown here rather than saved.
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "planetmodel[plot] @ "
         "git+https://github.com/da380/planetmodel@main"],
        check=True)
here = Path.cwd()
tutorials = here.parent if here.name == "notebooks" else here / "examples" / "tutorials"
__file__ = str(tutorials / "10_random_fields.py")
sys.argv = [__file__]
(tutorials.parent / "figures").mkdir(parents=True, exist_ok=True)
if "get_ipython" in globals():          # figures shown in the notebook, sharp
    get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")

In [ ]:
from pathlib import Path

import numpy as np

from planetmodel import (PREM, SCALAR, Geometry, Model, Skeleton, equiangular, slices,
                         testing)
from planetmodel.randomfield import LayeredGRF, RadialGRF, SphericalGRF

FIGURES = Path(__file__).resolve().parent.parent / "figures"
FIGURES.mkdir(exist_ok=True)
rng = np.random.default_rng(7)
model = PREM(ocean=False)
a = model.skeleton.boundaries[-1]

## A field of radius

`RadialGRF(r1, r2, nu, lam, sigma=...)` is a zero-mean Gaussian field
on the interval `[r1, r2]` with smoothness `nu`, length scale `lam`
and standard deviation `sigma`. The last two may be numbers or
functions of radius. A sample is the vector of the field's values at
the nodes of its own mesh, and the sampler is scaled so that the
standard deviation at every node is exactly the `sigma` asked for.
`to_field` turns a sample into a `RadialField` whose layer function is
the polynomial through the nodal values on each element. The example
is a field on the mantle whose standard deviation grows with radius.

In [ ]:
grf = RadialGRF(3480.0e3, a, 1.5, 400.0e3, sigma=lambda r: 0.02 * (r / a))
samples = grf.sample(rng=rng, size=3)
print(grf, "| samples:", samples.shape)
field = grf.to_field(samples[0], name="dv")
testing.check_field(field)
print("as a field at 5000 km:", field(5.0e6))

## The basis behind a sampler

A sampler is built on the eigenfunctions of its operator: functions
`phi_j` of radius that the operator merely multiplies by a number, the
eigenvalue `theta_j`. They are found on the sampler's mesh, ordered by
eigenvalue, and the smooth ones come first. A sample is a sum of them
with random coefficients, the rougher ones weighted less.

The mesh is longer than the interval asked for. The operator needs a
condition at each end of its mesh, and any condition distorts the
field near that end, so the mesh is extended by a *pad* on each side
and the distortion happens there, outside the interval that matters.
The interval asked for is called the physical interval.

`grf.basis` is the `SpectralBasis` the sampler was built from. It
holds the eigenvalues `theta` and gives the eigenfunctions on the
whole mesh (`modes`), or at any radius of the physical interval
(`evaluate`). A function is described by its coefficients in the
basis: `synthesise` turns coefficients into values at the nodes, and
`analyse` turns values on the whole mesh back into coefficients.
`weights` are the numbers that turn nodal values into an integral
over the physical interval, here against `r^2 dr`.

In [ ]:
basis = grf.basis
testing.check_spectral_basis(basis)
print(basis)
print("mesh:", grf.mesh.left[0] / 1e3, "to", grf.mesh.right[-1] / 1e3, "km;",
      "physical interval:", [x / 1e3 for x in basis.restriction.interval], "km")
coeffs = basis.white_noise(rng=rng) * basis.theta ** -grf.beta
values = basis.synthesise(coeffs, physical=False)
print("coefficients recovered from the values:",
      np.allclose(basis.analyse(values), coeffs))
r1, r2 = basis.restriction.interval
print("integral of r^2 over the mantle:", basis.weights().sum(),
      "against", (r2 ** 3 - r1 ** 3) / 3)
print("the first three modes at 5000 km:", basis.evaluate(5.0e6)[:3])

## A field on the layers of a model

`LayeredGRF` draws an independent field on each chosen layer and
returns one `RadialField` per layer of the model, zero on the layers
left out. The result is discontinuous across the layer boundaries.
Here PREM's density is given a random perturbation with a standard
deviation of two per cent on every layer outside the core, mantle and
crust alike, and none inside it. The perturbed densities are put back
into the model with `with_field`.

In [ ]:
mantle = [layer.index for layer in model.layers if layer.interval[0] >= 3480.0e3]
layered = LayeredGRF(model, 1.5, 300.0e3, sigma=0.02, layers=mantle, name="delta")
delta = layered.sample(rng=rng)
perturbed = model
for i in mantle:
    rho = model.layer(i)["rho"]
    perturbed = perturbed.with_field(i, "rho", rho * (1 + delta[i]), replace=True)
testing.check_model(perturbed)
r = np.array([2.0e6, 4.0e6, 6.0e6])
for x in r:
    i = model.skeleton.locate(x).layer
    print(f"r = {x / 1e3:.0f} km: delta = {delta[i](x):+.4f}, rho "
          f"{model.layer(i)['rho'](x):.1f} -> {perturbed.layer(i)['rho'](x):.1f}")

## A field on a spherical shell

`SphericalGRF` builds the same kind of field on a shell. A sample is
the set of real spherical-harmonic coefficients up to degree `lmax`,
each a function of radius. As for the radial field, the sampler is
scaled so that the standard deviation at any point of the truncated
field is exactly `sigma`. A horizontal length scale `lam_h`, different
from the radial one, gives structure that is correlated over a
different distance horizontally than radially. `to_field` turns a
sample into a `HarmonicField`, the sum of the harmonics with those
coefficients, exact at every point. The example is on a
shell of unit outer radius so that the picture is easy to read. The
standard deviation of a map at one radius is close to the requested
value of 1, as one draw from the field should be.

In [ ]:
shell = SphericalGRF(0.55, 1.0, 1.0, 0.06, lam_h=0.25, sigma=1.0, lmax=24)
coeffs = shell.sample(rng=rng)
print(shell, "| coefficients:", coeffs.shape)
random_field = shell.to_field(coeffs, character=SCALAR, name="dv")
testing.check_field(random_field)

# a model of the one shell holding the field, sliced on a sphere at one
# radius and through the centre (planetmodel.slices; tutorial 13)
shell_model = Model(Geometry(Skeleton([0.55, 1.0])), [{"dv": random_field}])
surface = slices.sphere(shell_model, "dv", radius=0.9, grid=equiangular(90, 180))
disc = slices.section(shell_model, "dv", (0.5 * np.pi, 0.0), azimuth=0.0,
                      length=2 * np.pi, n=60, m=361)
print("map std at r = 0.9:", surface.values.std().round(3), "(sigma = 1)")

`sample_grid` draws a sample directly on a Gauss-Legendre grid, as an
array of mesh nodes by grid points, using the fast spherical-harmonic
transforms of ducc0. The transforms themselves
are reached through `planetmodel.harmonics`. `analyse_grid` is the inverse
transform for a field band-limited at the grid's degree, and the two
round-trip to rounding error. The average of the square of one sample
over the sphere is an estimate of `sigma^2` from a single draw, and
scatters around 1.

In [ ]:
try:
    from planetmodel import gauss_legendre
    from planetmodel.harmonics import analyse_grid, synthesise_grid
    grid = gauss_legendre(shell.lmax)
    on_grid = shell.sample_grid(grid, rng=rng)
    back = analyse_grid(on_grid, grid)
    print("on the grid:", on_grid.shape, "| coefficients back:", back.shape,
          "| round trip error:", np.abs(synthesise_grid(back, grid) - on_grid).max())
    print("sphere average of u^2 at a node:",
          float(np.sum(on_grid[10] ** 2 * grid.weights[:, None])
                * (2 * np.pi / grid.nphi) / (4 * np.pi)).__round__(3), "(sigma^2 = 1)")
except ImportError as exc:
    print("skipped:", str(exc).splitlines()[0])

## The operator family underneath

`RadialOperatorFamily` holds the operator A = 1 - div(lambda^2 grad)
on the mesh as one matrix A_l for each spherical-harmonic degree l,
together with its mass matrix, its eigenvalues and eigenvectors, its
powers, its inverse and its white noise. These are the pieces from
which pygeoinf builds a function space, an operator on it and a
Gaussian measure. Here the smallest eigenvalues at degree 4, and a
check that applying A_4 undoes applying its inverse.

In [ ]:
family = shell.family
theta_l, Phi_l = family.eig(4)
print("degree 4:", family.ndof(4), "dofs, smallest eigenvalues",
      theta_l[:3].round(4))
noise = family.white_noise(4, rng=rng)
smooth = family.apply_power(4, -1.0, noise)
print("A_4^-1 white noise, then A_4 back:",
      np.abs(family.apply(4, smooth) - noise).max())

## The figure

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is not None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    ax = axes[0, 0]
    for s in samples:
        ax.plot(s, grf.r / 1e3, lw=0.8)
    sig = 0.02 * grf.r / a
    ax.plot(2 * sig, grf.r / 1e3, "k--", lw=0.8)
    ax.plot(-2 * sig, grf.r / 1e3, "k--", lw=0.8)
    ax.set_xlabel("sample value")
    ax.set_ylabel("radius [km]")
    ax.set_title("three radial samples and the 2 sigma envelope")

    ax = axes[0, 1]
    from planetmodel.plotting import radial_profile
    for m, color, label in ((model, "0.6", "PREM"),
                            (perturbed, "C0", "perturbed mantle")):
        radial_profile(ax, m, "rho", scale=1e-3, n=400, color=color, lw=0.9,
                       label=label)
    ax.set_xlabel("density [kg/m^3]")
    ax.set_ylabel("radius [km]")
    ax.legend()

    from planetmodel.plotting import section, sphere_map
    axes[1, 0].remove()
    axes[1, 1].remove()
    try:
        import cartopy.crs as ccrs
        ax = fig.add_subplot(2, 2, 3, projection=ccrs.Robinson())
        sphere_map(ax, surface, cmap="RdBu_r", vmin=-3, vmax=3, colorbar=True)
        ax.set_title("shell field at r = 0.9")
    except ImportError:
        print("the map needs the maps extra (cartopy); left out")

    ax = fig.add_subplot(2, 2, 4, projection="polar")
    section(ax, disc, cmap="RdBu_r", vmin=-3, vmax=3, colorbar=True)
    ax.set_rticks([0.6, 0.8, 1.0])
    ax.set_title("the plane through the poles")
    fig.tight_layout()

    # the first modes of the radial sampler on its padded mesh
    fig, ax = plt.subplots(figsize=(8, 4))
    x = grf.mesh.rglob / 1e3
    for j in range(5):
        ax.plot(x, basis.modes()[:, j], lw=0.9, label=f"mode {j}")
    ax.axvspan(r1 / 1e3, r2 / 1e3, color="0.9", label="physical interval")
    ax.set_xlabel("radius [km]")
    ax.set_ylabel("eigenfunction")
    ax.set_title("the first five modes; the pads lie outside the shaded band")
    ax.legend(fontsize=8)
    fig.tight_layout()